# Day-20: Agent Tools — Testing Notebook

## 1. Loading tools

In [11]:
import sys, os, json
sys.path.insert(0, "..")  # add agent-tools/ to path

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from tools.calculator     import execute as calc
from tools.web_search     import execute as search
from tools.database_query import execute as dbq
from tools.file_reader    import execute as fread
from tools.weather        import execute as weather
from tools.email_sender   import execute as email
from tools.datetime_tool  import execute as dt
from tools.data_analyzer  import execute as analyze
from tool_schemas import TOOLS_LIST, TOOLS_BY_NAME
from function_calling import run_agent

SAMPLE_DATA = "../sample_data"
print("\u2713 All tools loaded successfully")

✓ All tools loaded successfully


## 2. Schema Registry

In [12]:
print(f"Total tools registered: {len(TOOLS_LIST)}\n")
for schema in TOOLS_LIST:
    fn = schema["function"]
    props = fn["parameters"]["properties"]
    req = fn["parameters"].get("required", [])
    print(f"  {fn['name']:20s}  params={len(props):2d}  required={req}")

Total tools registered: 8

  calculator            params= 3  required=['operation', 'a']
  web_search            params= 2  required=['query']
  database_query        params= 5  required=[]
  file_reader           params= 2  required=['file_path']
  weather               params= 2  required=['city']
  email_sender          params= 5  required=['to', 'subject', 'body']
  datetime_tool         params= 4  required=['operation']
  data_analyzer         params= 8  required=['source']


## 3. Calculator Tests

In [13]:
tests = [
    ("add",        {"operation": "add",        "a": 15,  "b": 27}),
    ("subtract",   {"operation": "subtract",   "a": 100, "b": 43}),
    ("multiply",   {"operation": "multiply",   "a": 6,   "b": 7}),
    ("divide",     {"operation": "divide",     "a": 22,  "b": 7}),
    ("power",      {"operation": "power",      "a": 2,   "b": 10}),
    ("sqrt",       {"operation": "sqrt",       "a": 144}),
    ("percentage", {"operation": "percentage", "a": 15,  "b": 847}),
    ("div by 0",   {"operation": "divide",     "a": 5,   "b": 0}),   # error
]

print(f"{'Operation':<15} {'Args':<30} {'Result'}")
print("-"*60)
for label, args in tests:
    r = calc(args)
    if r["success"]:
        dash = '\u2014'
        b_val = args.get('b', dash)
        print(f"  {label:<13} a={args.get('a')}, b={b_val:<10}  → {r['result']}")
    else:
        print(f"  {label:<13} ERROR: {r['error']}")

Operation       Args                           Result
------------------------------------------------------------
  add           a=15, b=27          → 42.0
  subtract      a=100, b=43          → 57.0
  multiply      a=6, b=7           → 42.0
  divide        a=22, b=7           → 3.1428571429
  power         a=2, b=10          → 1024.0
  sqrt          a=144, b=—           → 12.0
  percentage    a=15, b=847         → 127.05
  div by 0      ERROR: Division by zero is undefined.


## 4. Web Search Test

In [14]:
result = search({"query": "OpenAI function calling Python tutorial", "max_results": 3})
if result["success"]:
    print(f"Query: {result['query']}")
    print(f"Results: {result['num_results']}\n")
    for r in result["results"]:
        print(f"  \u2022 {r['title']}")
        print(f"    {r['url']}")
        print(f"    {r['snippet'][:100]}...\n")
else:
    print(f"Error: {result['error']}")

Error: Search library not installed. Run: pip install ddgs


## 5. Database Query Tests

In [15]:
# All students (limit 5)
r = dbq({"limit": 5})
print(f"All students (limit 5): {r.get('row_count', 'N/A')} rows")
for row in r.get("rows", []):
    print(f"  {row}")

print()

# Filter by course
r2 = dbq({"filter_by": "course", "filter_value": "Mathematics", "limit": 3})
if r2["success"]:
    print(f"Mathematics students: {r2['row_count']} found")
    for row in r2.get("rows", [])[:3]:
        print(f"  {row}")
else:
    print(f"Filter result: {r2}")

print()

# Top 5 by marks
r3 = dbq({"order_by": "marks DESC", "limit": 5})
print(f"Top 5 by marks:")
for row in r3.get("rows", []):
    print(f"  {row.get('name'):15s}  marks={row.get('marks')}")

print()

# Aggregates
for agg in ["count", "avg_marks", "max_marks", "min_marks"]:
    ra = dbq({"aggregate": agg})
    if ra["success"]:
        print(f"  {agg}: {ra['result']}")

All students (limit 5): N/A rows

Filter result: {'success': False, 'error': 'psycopg2 not installed. Run: pip install psycopg2-binary'}

Top 5 by marks:



## 6. File Reader Tests

In [16]:
for fname, extra in [
    ("sample.txt",  {"max_lines": 5}),
    ("sample.csv",  {"max_lines": 4}),
    ("sample.json", {}),
]:
    path = os.path.join(SAMPLE_DATA, fname)
    r = fread({"file_path": path, **extra})
    print(f"{'='*40}")
    print(f"File: {fname}  (type={r.get('file_type','?')})")
    if r["success"]:
        if r["file_type"] == "txt":
            print(f"  Lines: {r['shown_lines']}/{r['total_lines']}")
            for line in r["content"][:3]:
                print(f"    {line[:60]}")
        elif r["file_type"] == "csv":
            print(f"  Columns: {r['columns']}")
            print(f"  Rows shown: {r['shown_rows']}/{r['total_rows']}")
            for row in r["rows"]:
                print(f"    {row}")
        elif r["file_type"] == "json":
            print(f"  Type: {r['data_type']}, length: {r['length']}")
            if isinstance(r["content"], dict):
                for k, v in list(r["content"].items())[:4]:
                    print(f"    {k}: {v}")
    else:
        print(f"  ERROR: {r['error']}")
    print()

File: sample.txt  (type=?)
  ERROR: File not found: '../sample_data/sample.txt'.

File: sample.csv  (type=?)
  ERROR: File not found: '../sample_data/sample.csv'.

File: sample.json  (type=?)
  ERROR: File not found: '../sample_data/sample.json'.



## 7. Weather Tests

In [17]:
cities = [
    ("London",   "celsius"),
    ("Mumbai",   "celsius"),
    ("New York", "fahrenheit"),
]

print(f"{'City':<15} {'Temp':<10} {'Humidity':<10} {'Wind':<12} {'Condition'}")
print("-"*65)
for city, units in cities:
    r = weather({"city": city, "units": units})
    if r["success"]:
        print(f"  {r['city']:<13} {str(r['temperature'])+r['temperature_unit']:<10} "
              f"{str(r['humidity_percent'])+'%':<10} {str(r['wind_speed_kmh'])+' kmh':<12} "
              f"{r['condition']}")
    else:
        print(f"  {city:<13} ERROR: {r['error']}")

City            Temp       Humidity   Wind         Condition
-----------------------------------------------------------------
  London        16.1°C     54%        16.9 kmh     Overcast
  Mumbai        29.8°C     79%        16.6 kmh     Overcast
  New York      67.3°F     78%        9.1 kmh      Overcast


## 8. Email Sender (Simulation)

In [18]:
r1 = email({
    "to": "alice@example.com",
    "subject": "Meeting Reminder",
    "body": "Hi Alice, reminder about our 10am meeting tomorrow.",
})
print(f"Email 1: message_id={r1.get('message_id')}, logged_to={r1.get('logged_to')}")

r2 = email({
    "to": "cto@company.com",
    "subject": "URGENT: Database Alert",
    "body": "High CPU detected on production DB. Investigating.",
    "cc": "devops@company.com",
    "priority": "high",
})
print(f"Email 2: message_id={r2.get('message_id')}, priority={r2.get('priority')}")

# Show last 2 log entries
log_path = r1.get("logged_to", "")
if log_path and os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print(f"\nLast {min(2, len(lines))} entries in email_log.txt:")
    for line in lines[-2:]:
        entry = json.loads(line)
        print(f"  {entry['message_id']}  to={entry['to']}  subject={entry['subject'][:40]}")


📧 [EMAIL SIMULATED]
   To      : alice@example.com
   Subject : Meeting Reminder
   Priority: NORMAL
   ID      : MSG-20260417121123-3663
   Logged  : /Users/pradhnyesh/Documents/Linkific-training/Day-20/agent-tools/tools/../logs/email_log.txt

Email 1: message_id=MSG-20260417121123-3663, logged_to=/Users/pradhnyesh/Documents/Linkific-training/Day-20/agent-tools/logs/email_log.txt

🚨 [EMAIL SIMULATED]
   To      : cto@company.com
   CC      : devops@company.com
   Subject : URGENT: Database Alert
   Priority: HIGH
   ID      : MSG-20260417121123-4984
   Logged  : /Users/pradhnyesh/Documents/Linkific-training/Day-20/agent-tools/tools/../logs/email_log.txt

Email 2: message_id=MSG-20260417121123-4984, priority=high

Last 2 entries in email_log.txt:
  MSG-20260417121123-3663  to=alice@example.com  subject=Meeting Reminder
  MSG-20260417121123-4984  to=cto@company.com  subject=URGENT: Database Alert


## 9. Datetime Tests

In [19]:
operations = [
    ("now",           {}),
    ("add_days",      {"date_str": "2026-04-17", "days": 30}),
    ("subtract_days", {"date_str": "2026-04-17", "days": 7}),
    ("day_of_week",   {"date_str": "2026-01-01"}),
    ("format_date",   {"date_str": "2026-04-17", "format_str": "%B %d, %Y"}),
    ("days_between",  {"date_str": "2026-01-01,2026-04-17"}),
    ("is_weekend",    {"date_str": "2026-04-19"}),
]

for op, extra in operations:
    r = dt({"operation": op, **extra})
    if r["success"]:
        # Show the most interesting key
        keys = [k for k in r.keys() if k not in ("success", "operation")]
        preview = {k: r[k] for k in keys[:3]}
        print(f"  {op:<15}  {preview}")
    else:
        print(f"  {op:<15}  ERROR: {r['error']}")

  now              {'datetime': '2026-04-17T12:11:23Z', 'date': '2026-04-17', 'time': '12:11:23'}
  add_days         {'original_date': '2026-04-17', 'days': 30, 'result_date': '2026-05-17'}
  subtract_days    {'original_date': '2026-04-17', 'days': 7, 'result_date': '2026-04-10'}
  day_of_week      {'date': '2026-01-01', 'weekday': 'Thursday', 'weekday_num': 3}
  format_date      {'date': '2026-04-17', 'format_str': '%B %d, %Y', 'result': 'April 17, 2026'}
  days_between     {'date1': '2026-01-01', 'date2': '2026-04-17', 'days': 106}
  is_weekend       {'date': '2026-04-19', 'weekday': 'Sunday', 'is_weekend': True}


## 10. Data Analyzer Tests

In [20]:
csv_path = os.path.join(SAMPLE_DATA, "sample.csv")

for op, extra in [
    ("describe",     {}),
    ("value_counts", {"column": "course"}),
    ("groupby",      {"group_by": "course"}),
    ("filter",       {"filter_column": "course", "filter_value": "Math"}),
    ("top_n",        {"top_n": 3}),
    ("correlation",  {}),
]:
    r = analyze({"source": csv_path, "operation": op, **extra})
    if r["success"]:
        result_preview = str(r["result"])[:80]
        print(f"  [{op}] shape={r['shape']}  result={result_preview}...")
    else:
        print(f"  [{op}] ERROR: {r['error']}")
    print()

  [describe] ERROR: pandas not installed. Run: pip install pandas

  [value_counts] ERROR: pandas not installed. Run: pip install pandas

  [groupby] ERROR: pandas not installed. Run: pip install pandas

  [filter] ERROR: pandas not installed. Run: pip install pandas

  [top_n] ERROR: pandas not installed. Run: pip install pandas

  [correlation] ERROR: pandas not installed. Run: pip install pandas



## 11. Single-Tool Agent Calls

In [21]:
agent_tokens = []

questions = [
    "What is the square root of 256?",
    "What is the current weather in Paris?",
    "What day of the week is 2026-12-25?",
    "How many students are in the database?",
]

for q in questions:
    r = run_agent(q, verbose=True)
    agent_tokens.append(r["total_tokens"])
    print(f"\n  Tools called: {[c['tool'] for c in r['tool_calls_made']]}")
    print()


Q: What is the square root of 256?

  [TOOL CALL] calculator
  [ARGS]      {"operation": "sqrt", "a": 256}
  [RESULT]    ✓ {"success": true, "operation": "sqrt", "a": 256.0, "b": null, "result": 16.0}

A: The square root of 256 is 16.
   [iterations=2, tokens=2208]

  Tools called: ['calculator']


Q: What is the current weather in Paris?

  [TOOL CALL] weather
  [ARGS]      {"city": "Paris"}
  [RESULT]    ✓ {"success": true, "city": "Paris", "country": "France", "latitude": 48.85341, "longitude": 2.3488, "temperature": 20.2, "temperature_unit": "\u00b0C", "humidity_percent": 42, "wind_speed_kmh": 0.8, "c

A: The current weather in Paris is partly cloudy with a temperature of 20.2°C. The humidity is at 42%, and there is a light wind blowing at a speed of 0.8 km/h.
   [iterations=2, tokens=2285]

  Tools called: ['weather']


Q: What day of the week is 2026-12-25?

  [TOOL CALL] datetime_tool
  [ARGS]      {"operation": "day_of_week", "date_str": "2026-12-25"}
  [RESULT]    ✓ {"success

## 12. Multi-Tool Chaining

In [23]:
chaining_questions = [
    "What is the current weather in Tokyo AND what day of the week is 2026-04-17?",
    "How many students are in the database, and also calculate what 30 is as a percentage of 200?",
]

for q in chaining_questions:
    r = run_agent(q, verbose=True)
    agent_tokens.append(r["total_tokens"])
    arrow = " \u2192 "
    chain = arrow.join(c['tool'] for c in r['tool_calls_made'])
    print(f"\n  Chain: {chain}")
    print(f"  Total tool calls: {len(r['tool_calls_made'])}")
    print()


Q: What is the current weather in Tokyo AND what day of the week is 2026-04-17?

  [TOOL CALL] weather
  [ARGS]      {"city": "Tokyo"}
  [RESULT]    ✓ {"success": true, "city": "Tokyo", "country": "Japan", "latitude": 35.6895, "longitude": 139.69171, "temperature": 12.2, "temperature_unit": "\u00b0C", "humidity_percent": 69, "wind_speed_kmh": 3.9, "

  [TOOL CALL] datetime_tool
  [ARGS]      {"operation": "day_of_week", "date_str": "2026-04-17"}
  [RESULT]    ✓ {"success": true, "operation": "day_of_week", "date": "2026-04-17", "weekday": "Friday", "weekday_num": 4}

A: The current weather in Tokyo is partly cloudy with a temperature of 12.2°C, humidity at 69%, and a wind speed of 3.9 km/h.

The day of the week for April 17, 2026, is Friday.
   [iterations=2, tokens=2444]

  Chain: weather → datetime_tool
  Total tool calls: 2


Q: How many students are in the database, and also calculate what 30 is as a percentage of 200?

  [TOOL CALL] database_query
  [ARGS]      {"aggregate": "cou

## 13. Agent Error Handling

In [24]:
error_questions = [
    "Get the weather for the city called 'Xyzabc123notacity'.",
    "Read the file at /this/does/not/exist.txt and tell me what's in it.",
]

for q in error_questions:
    r = run_agent(q, verbose=True)
    agent_tokens.append(r["total_tokens"])
    if r["tool_calls_made"]:
        tool_result = r["tool_calls_made"][0]["result"]
        print(f"\n  Tool returned error: {tool_result.get('error')}")
        print(f"  Agent explained: {r['answer'][:150]}")
    print()


Q: Get the weather for the city called 'Xyzabc123notacity'.

  [TOOL CALL] weather
  [ARGS]      {"city": "Xyzabc123notacity"}
  [RESULT]    ✗ {"success": false, "error": "City not found: 'Xyzabc123notacity'. Try a more specific name."}

A: I couldn't find any weather information for the city named "Xyzabc123notacity." It seems to be an invalid or non-existent city name. If you have another city in mind, please let me know!
   [iterations=2, tokens=2249]

  Tool returned error: City not found: 'Xyzabc123notacity'. Try a more specific name.
  Agent explained: I couldn't find any weather information for the city named "Xyzabc123notacity." It seems to be an invalid or non-existent city name. If you have anoth


Q: Read the file at /this/does/not/exist.txt and tell me what's in it.

  [TOOL CALL] file_reader
  [ARGS]      {"file_path": "/this/does/not/exist.txt"}
  [RESULT]    ✗ {"success": false, "error": "File not found: '/this/does/not/exist.txt'."}

A: The file at the specified path `

## 14. Token Usage Summary

In [25]:
PRICING = {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000}

total_tokens = sum(agent_tokens)
# Rough estimate: ~70% input, 30% output
est_cost = (total_tokens * 0.7 * PRICING["input"]) + (total_tokens * 0.3 * PRICING["output"])

print(f"Agent calls made  : {len(agent_tokens)}")
print(f"Total tokens used : {total_tokens:,}")
print(f"Estimated cost    : ${est_cost:.6f} USD (gpt-4o-mini pricing)")
print()
print("Per-call breakdown:")
for i, tok in enumerate(agent_tokens, 1):
    print(f"  Call {i:2d}: {tok:5,} tokens")

Agent calls made  : 8
Total tokens used : 18,289
Estimated cost    : $0.005212 USD (gpt-4o-mini pricing)

Per-call breakdown:
  Call  1: 2,208 tokens
  Call  2: 2,285 tokens
  Call  3: 2,243 tokens
  Call  4: 2,236 tokens
  Call  5: 2,444 tokens
  Call  6: 2,386 tokens
  Call  7: 2,249 tokens
  Call  8: 2,238 tokens
